In [ ]:
import pandas as pd

roll_no = input("Enter your college roll number: ")

last_two = roll_no[-2:]
digits = [int(d) for d in last_two]

categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = []

for d in digits:
    category = categories[d % 3]

    if category == "billing":
        question = "how can i check my payment status"
        answer = "You can check your payment status from the billing section."
        keywords = "payment status billing transaction"
    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "Go to Account Settings and update your registered mobile number."
        keywords = "mobile number update account"
    else:
        question = "where can i get general help"
        answer = "You can contact customer support for general assistance."
        keywords = "help support assistance"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

df = pd.DataFrame(fixed_entries + personalized_entries)

print("Final 6-row DataFrame:")
display(df)

def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = row["question"].lower() + " " + row["keywords"].lower()
        text_words = set(text.split())
        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    return sorted(results, key=lambda x: x["confidence"], reverse=True)

query = input("Enter your query: ")
results = score_query(query, df)

print("\nMatching entries:")
if results:
    display(pd.DataFrame(results))
else:
    print("No matching FAQ found.")

def same_category(category_name, df):
    return df[df["category"] == category_name]

personalized_category = personalized_entries[0]["category"]

print("\nQ3: Entries belonging to category:", personalized_category)
display(same_category(personalized_category, df)[["question", "category"]])

print("\nQ4: Choose an FAQ entry")
for i in range(len(df)):
    print(i, "-", df.loc[i, "question"])

choice = int(input("Enter entry number: "))
new_keyword = input("Enter a new keyword: ")

df.loc[choice, "keywords"] = df.loc[choice, "keywords"] + " " + new_keyword

print("\nUpdated entry:")
display(df.loc[[choice]])

filename = roll_no + "_faq_data.csv"
df.to_csv(filename, index=False)

print("Updated DataFrame saved as:", filename)

print("\nQ5: Number of FAQ entries per category:")
display(df.groupby("category").size().reset_index(name="count"))

def improved_score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = row["question"].lower() + " " + row["keywords"].lower()
        text_words = set(text.split())
        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    return sorted(results, key=lambda x: x["confidence"], reverse=True)

tie_query = "fee"
tie_results = improved_score_query(tie_query, df)

print("\nQ6: Tie demonstration")
print("Query:", tie_query)

if tie_results:
    highest_score = tie_results[0]["confidence"]
    top_matches = [r for r in tie_results if r["confidence"] == highest_score]

    if len(top_matches) > 1:
        print("Tie found. All matching entries:")
    else:
        print("Best matching entry:")

    display(pd.DataFrame(top_matches))

non_tie_query = "password reset"
non_tie_results = improved_score_query(non_tie_query, df)

print("\nNon-tie demonstration")
print("Query:", non_tie_query)

if non_tie_results:
    highest_score = non_tie_results[0]["confidence"]
    top_matches = [r for r in non_tie_results if r["confidence"] == highest_score]

    if len(top_matches) > 1:
        print("Tie found. All matching entries:")
    else:
        print("No tie. Best matching entry:")

    display(pd.DataFrame(top_matches))
